# most_promising_learn

This notebook is the closest honest replay of the most promising deep_orderbook branch found so far.

Chosen branch:
- mapper: `regonly_l1_2d_h8`
- market: `BTC-USD`
- cadence: `1000ms`
- geometry: `rolling_window_size=1024`, `look_ahead=64`, `window_stride=4`
- train pool: recent 2026 files from `2026-04-16..2026-04-21` at `00/06/12/18`
- direction: `short`
- route family: fixed-route stress-conditioned replay
- honest reference checkpoint: `exp57_recent24_stress_short_0600_0700_route_falsification_corrected_friction`

Why this one:
- `exp54` had the biggest raw PnL but later friction accounting was shown to be unreliable there.
- `exp57` is the strongest corrected-friction result I trust: positive cost-adjusted PnL at cost-5 while keeping `rmse_ratio < 1`.
- Latest continuation (`exp58`, `exp59`) degraded under friction, so this is still a research replay, not a production strategy.

What this notebook does:
1. exposes the exact config
2. runs the same mapper/route search on chosen holdouts
3. writes the usual JSON + PNG artifacts under `experiments/`

Important caveat:
- the top chart can hide short prediction trades because the current plotting only marks `0->1` and `1->0` transitions, while short positions are encoded as `-1`.


In [ ]:
from pathlib import Path
import json
from pprint import pprint

from deep_orderbook.config import ReplayConfig
from scripts.exp16_batch_h64_tcn import DEFAULT_SHAPER_CONFIG_2026
from scripts.exp22_trigger_sweep import main as exp22_main
from deep_orderbook.btc_search_lab import get_batch_variant


In [ ]:
DATA_DIR = Path('/mnt/data/repos/gaelreinaudi/crypto')

VARIANT = 'regonly_l1_2d_h8'
DIRECTIONS = ['short']
LABEL = 'most_promising_learn'
N_SAMPLES_PER_FILE = None  # uncapped, same honest setting as the serious runs
LOAD_EVERY = '1000ms'

TRAIN_FILES = [
    DATA_DIR / '2026-04-16T00-00-00.parquet',
    DATA_DIR / '2026-04-16T06-00-01.parquet',
    DATA_DIR / '2026-04-16T12-00-01.parquet',
    DATA_DIR / '2026-04-16T18-00-01.parquet',
    DATA_DIR / '2026-04-17T00-00-01.parquet',
    DATA_DIR / '2026-04-17T06-00-01.parquet',
    DATA_DIR / '2026-04-17T12-00-00.parquet',
    DATA_DIR / '2026-04-17T18-00-11.parquet',
    DATA_DIR / '2026-04-18T00-00-01.parquet',
    DATA_DIR / '2026-04-18T06-00-00.parquet',
    DATA_DIR / '2026-04-18T12-00-00.parquet',
    DATA_DIR / '2026-04-18T18-00-00.parquet',
    DATA_DIR / '2026-04-19T00-00-00.parquet',
    DATA_DIR / '2026-04-19T06-00-00.parquet',
    DATA_DIR / '2026-04-19T12-00-00.parquet',
    DATA_DIR / '2026-04-19T18-00-00.parquet',
    DATA_DIR / '2026-04-20T00-00-04.parquet',
    DATA_DIR / '2026-04-20T06-00-00.parquet',
    DATA_DIR / '2026-04-20T12-00-00.parquet',
    DATA_DIR / '2026-04-20T18-00-01.parquet',
    DATA_DIR / '2026-04-21T00-00-00.parquet',
    DATA_DIR / '2026-04-21T06-00-00.parquet',
    DATA_DIR / '2026-04-21T12-00-00.parquet',
    DATA_DIR / '2026-04-21T18-00-10.parquet',
]

# The most honest corrected-friction checkpoint I trust most is exp57.
# You can swap these holdouts if you want to replay a different continuation.
TEST_FILES = [
    DATA_DIR / '2026-04-23T06-00-00.parquet',
    DATA_DIR / '2026-04-23T07-00-00.parquet',
]

SHAPER_CONFIG = DEFAULT_SHAPER_CONFIG_2026.but(
    rolling_window_size=1024,
    look_ahead=64,
    window_stride=4,
)

REPLAY_TEMPLATE = ReplayConfig(
    markets=['BTC-USD'],
    data_dir=DATA_DIR,
    every=LOAD_EVERY,
)

VARIANT_CONFIG = get_batch_variant(VARIANT)

print('Variant config:')
pprint(VARIANT_CONFIG)
print('Train files:', len(TRAIN_FILES))
print('Test files:', len(TEST_FILES))
print('Shaper config:')
pprint(SHAPER_CONFIG.model_dump())


In [ ]:
# In Jupyter, top-level await is fine.
await exp22_main(
    variants=[VARIANT],
    train_files=TRAIN_FILES,
    test_files=TEST_FILES,
    label=LABEL,
    directions=DIRECTIONS,
    n_samples_per_file=N_SAMPLES_PER_FILE,
    load_every=LOAD_EVERY,
    shaper_config_override=SHAPER_CONFIG,
)


In [ ]:
from pathlib import Path
import json

results_dir = Path('experiments/results')
pictures_dir = Path('experiments/pictures')

recent_json = sorted(results_dir.glob(f'{LABEL}*.json'))[-10:]
recent_png = sorted(pictures_dir.glob(f'{LABEL}*.png'))[-10:]

print('Recent JSON artifacts:')
for p in recent_json:
    print('-', p)

print('Recent PNG artifacts:')
for p in recent_png:
    print('-', p)

if recent_json:
    latest = recent_json[-1]
    print('Latest JSON summary:', latest)
    data = json.loads(latest.read_text())
    if 'aggregate_results' in data:
        pprint(data['aggregate_results'])
    else:
        pprint({k: data.get(k) for k in ['strategy_name', 'direction', 'route_score', 'rmse_ratio', 'metrics', 'strategy_metrics']})


## Practical replay notes

- If you want the strongest raw-PnL branch instead of the strongest corrected-friction branch, inspect `exp54`, but do not trust its friction survival without recomputing costs.
- If you want to continue the chronology, change `TEST_FILES` to later fresh 2026 files and rerun.
- If you want the top chart to show short prediction trades, `deep_orderbook/visu.py` needs to handle `0->-1` and `-1->0` marker transitions.
